# 10 — Fusion multimodale (RQ1) et robustesse (RQ3)

Sur les **112 participants** qui ont l'imagerie, on compare le **meme modele**
(random forest) nourri avec trois jeux de variables :

- **tabulaire** seul (clinique + cognitif + biomarqueurs) ;
- **imagerie** seule (IRM, PET, puis IRM+PET) ;
- **fusion** (tabulaire + imagerie).

Tache : **progression** (declin du diagnostic). Validation croisee 5 plis,
metrique principale ROC-AUC.

- **RQ1** : la fusion fait-elle mieux que le meilleur modele unimodal ?
- **RQ3** : que devient la fusion si une modalite est **absente** au test ?

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.append(r"..\configs")
from config_projet import C

# --- CHOIX DES DONNEES -------------------------------------------------------------
# "initiales" : notebooks 07 + 09 (cible 'progression' decalee dans le temps, images non alignees)
# "corrigees" : notebooks 07b + 09b (cible 'progression_image' ancree sur la date de l'image,
#               age rempli, images alignees sur le cerveau de reference MNI)
DONNEES = "corrigees"
# (donnees corrigees) True : on ne garde que les patients pas encore dements a la date de l'image,
# car un patient deja au stade maximal ne peut pas s'aggraver.
EXCLURE_DEJA_DEMENTS = True
SUFFIXE = "_corrigee" if DONNEES == "corrigees" else ""

if DONNEES == "initiales":
    prep = sorted(C.COHORTE.glob("preparation_modele_*"))[-1]          # notebook 07
    feat = sorted(C.COHORTE.glob("features_image_[0-9]*"))[-1]         # notebook 09
    jeu = pd.read_csv(prep / "jeu_modelisation_359.csv")
    images = pd.read_csv(feat / "features_image.csv")
    CIBLE, DX_DEPART = "progression", "dx_debut"
else:
    prep = sorted(C.COHORTE.glob("cible_ancree_image_*"))[-1]          # notebook 07b
    feat = sorted(C.COHORTE.glob("features_image_mni_*"))[-1]          # notebook 09b
    jeu = pd.read_csv(prep / "jeu_modelisation_112_ancre.csv")
    images = pd.read_csv(feat / "features_regions.csv")
    CIBLE, DX_DEPART = "progression_image", "dx_image"
print("Donnees :", DONNEES, "| tabulaire :", prep.name, "| imagerie :", feat.name)

jeu["PTID"] = jeu["PTID"].astype(str).str.strip().str.upper()
images["PTID"] = images["PTID"].astype(str).str.strip().str.upper()

# Fusionner : on ne garde que les participants qui ont l'imagerie (112).
donnees = jeu.merge(images, on="PTID", how="inner", validate="one_to_one")
print("Participants avec tabulaire ET imagerie :", len(donnees))

In [ ]:
# Definition des jeux de variables.
TAB_CANDIDATES = [
    "sexe", "education", "age_reference", "APOE_e4",
    "CDR_global", "CDR_SB", "MMSE_debut", "ADAS13", "FAQ_debut",
    "ABETA42", "TAU", "PTAU",
    DX_DEPART,   # diagnostic de depart : dx_debut (initiales) ou dx_image (corrigees)
]
if DONNEES == "initiales":
    # 5 statistiques par modalite (notebook 09).
    IRM_COLS = ["IRM_moyenne", "IRM_ecart_type", "IRM_p10", "IRM_p50", "IRM_p90"]
    PET_COLS = ["PET_moyenne", "PET_ecart_type", "PET_p10", "PET_p50", "PET_p90"]
else:
    # Mesures par region du cerveau (notebook 09b) : part de liquide / matiere grise (IRM),
    # activite relative (PET) dans l'hippocampe, l'amygdale, les ventricules, le lobe temporal...
    IRM_COLS = [c for c in images.columns if c.startswith("IRM_") and (c.endswith("_LCR") or c.endswith("_MG"))]
    PET_COLS = [c for c in images.columns if c.startswith("PET_")
                and c not in ("PET_corr_matiere_grise", "PET_essais")]

# Retirer les colonnes tabulaires entierement vides.
TAB = [c for c in TAB_CANDIDATES if donnees[c].notna().sum() > 0]
vides = [c for c in TAB_CANDIDATES if c not in TAB]
if vides:
    print("Variables tabulaires ignorees (vides) :", vides)

IMG = IRM_COLS + PET_COLS
FUSION = TAB + IMG
print("Variables : tabulaire", len(TAB), "| IRM", len(IRM_COLS), "| PET", len(PET_COLS))

# Cible : progression.
donnees = donnees.dropna(subset=[CIBLE]).copy()
if DONNEES == "corrigees" and EXCLURE_DEJA_DEMENTS:
    n0 = len(donnees)
    donnees = donnees.loc[donnees["deja_dement"] == 0].copy()
    print("Patients deja dements a la date de l'image exclus :", n0 - len(donnees))
y = donnees[CIBLE].astype(int)
print("Participants :", len(y), "| progressent :", int(y.sum()),
      f"({y.mean() * 100:.1f}%)")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

SCORES = ["roc_auc", "balanced_accuracy", "f1", "accuracy"]
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

def modele():
    return Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=300, class_weight="balanced", random_state=0)),
    ])

def evaluer(colonnes):
    res = cross_validate(modele(), donnees[colonnes], y, cv=CV, scoring=SCORES)
    return {s: res["test_" + s].mean() for s in SCORES}

In [ ]:
# --- RQ1 : comparaison des jeux de variables ---
jeux = {
    "Tabulaire seul": TAB,
    "Imagerie IRM seule": IRM_COLS,
    "Imagerie PET seule": PET_COLS,
    "Imagerie IRM+PET": IMG,
    "Fusion (tabulaire + imagerie)": FUSION,
}

tableau = pd.DataFrame(
    {nom: evaluer(cols) for nom, cols in jeux.items()}
).T.round(3)
tableau.index.name = "modele"

print(f"=== RQ1 — Progression, {len(y)} participants ({DONNEES}) ===")
display(tableau)

meilleur_uni = tableau.drop("Fusion (tabulaire + imagerie)")["roc_auc"].max()
auc_fusion = tableau.loc["Fusion (tabulaire + imagerie)", "roc_auc"]
print(f"Meilleur unimodal : AUC {meilleur_uni:.3f}")
print(f"Fusion            : AUC {auc_fusion:.3f}")
print("La fusion apporte :", round(auc_fusion - meilleur_uni, 3), "d'AUC")

In [ ]:
# --- RQ3 : robustesse quand une modalite manque au test ---
# On entraine le modele de FUSION, puis on evalue en remplacant les
# variables d'une modalite par la mediane d'entrainement (modalite "absente").
from sklearn.metrics import roc_auc_score

def auc_avec_masque(colonnes_masquees):
    aucs = []
    for app, test in CV.split(donnees[FUSION], y):
        X_app, X_test = donnees[FUSION].iloc[app], donnees[FUSION].iloc[test].copy()
        y_app, y_test = y.iloc[app], y.iloc[test]

        m = modele().fit(X_app, y_app)
        mediane = X_app.median(numeric_only=True)
        for col in colonnes_masquees:
            X_test[col] = mediane[col]   # modalite absente -> valeur neutre

        proba = m.predict_proba(X_test)[:, 1]
        aucs.append(roc_auc_score(y_test, proba))
    return float(np.mean(aucs))

robustesse = pd.DataFrame({
    "situation": [
        "Fusion complete",
        "Imagerie absente (tabulaire seul au test)",
        "Tabulaire absent (imagerie seule au test)",
    ],
    "AUC": [
        auc_avec_masque([]),
        auc_avec_masque(IMG),
        auc_avec_masque(TAB),
    ],
}).round(3)

print("=== RQ3 — Robustesse du modele de fusion ===")
display(robustesse)

In [ ]:
# Importance des variables dans le modele de fusion (indicatif).
Xi = SimpleImputer(strategy="median").fit_transform(donnees[FUSION])
rf = RandomForestClassifier(
    n_estimators=300, class_weight="balanced", random_state=0).fit(Xi, y)
importance = (pd.Series(rf.feature_importances_, index=FUSION)
              .sort_values(ascending=False).round(3))
print("Importance des variables (fusion) :")
display(importance.to_frame("importance"))

# Enregistrer les resultats.
SORTIE = C.sortie("fusion_RQ1_RQ3" + SUFFIXE)
tableau.to_csv(SORTIE / "RQ1_comparaison.csv", encoding="utf-8-sig")
robustesse.to_csv(SORTIE / "RQ3_robustesse.csv", index=False, encoding="utf-8-sig")
importance.to_frame("importance").to_csv(SORTIE / "importance_fusion.csv", encoding="utf-8-sig")
print("Resultats enregistres dans :", SORTIE)